# 08 — Bronze: Receita Federal — Empresas

## Purpose
Ingests Receita Federal Empresas data (10 shards) into the Bronze layer.
Processes each shard sequentially and appends to a single Parquet per month.

## Input
- `data/raw/receita_federal/{YYYY-MM}/Empresas{0-9}.zip` — 10 shards per month
- Sample month: ~27M total rows across all shards

## Output
- `data/bronze/rf_empresas/_year_month={YYYY-MM}/data.parquet` — single file per month
- Schema: 7 source columns (all STRING) + 5 technical columns

## Steps
1. Imports and configuration
2. Schema definition
3. Process shards
4. Validation
5. Ops registration

## Notes
- No headers in source CSV — columns assigned from official RF layout (positional)
- Encoding: latin-1 with fallback to utf-8, cp1252 (RF encoding is inconsistent)
- Pandas used for ZIP/CSV reading only — encoding fallback not available in DuckDB
- DuckDB receives the DataFrame via `con.register()` and handles Parquet write
- Shards processed sequentially — each appended to same Parquet via atomic os.replace()
- `_rescued_data`: schema drift for positional CSVs = unexpected column count
- Idempotent — deletes existing output before processing shards
- ADR-002: all source columns as STRING


## Step 1 — Imports and configuration

In [ ]:
import io
import json
import os
import sys
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd  # used only in read_zip_csv — encoding fallback

sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import get_project_root, to_sql_path
from duckdb_utils import get_connection
from validation import CheckSuite
from bootstrap_log import append_log, make_entry
from pipeline import check_landing_gate

# ---------------------------------------------------------------------------
# Project root and paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = get_project_root()

RF_ROOT     = PROJECT_ROOT / "data" / "raw"    / "receita_federal"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze" / "rf_empresas"
DRIFT_LOG   = PROJECT_ROOT / "db"  / "schema_drift_log.json"
LOG_PATH    = PROJECT_ROOT / "db"  / "bootstrap_log.json"

# ---------------------------------------------------------------------------
# Landing gate check
# ---------------------------------------------------------------------------
check_landing_gate(PROJECT_ROOT)

# ---------------------------------------------------------------------------
# Month selection — always processes the most recent available month
# ---------------------------------------------------------------------------
available_months = sorted([d.name for d in RF_ROOT.iterdir() if d.is_dir()])
if not available_months:
    raise FileNotFoundError(
        f"No month directories found in {RF_ROOT}\n"
        "Run 01_bootstrap_receita_federal.py first."
    )

SAMPLE_MONTH = available_months[-1]
SAMPLE_DIR   = RF_ROOT / SAMPLE_MONTH

# ---------------------------------------------------------------------------
# Pipeline metadata
# ---------------------------------------------------------------------------
SOURCE_ID  = "receita_federal"
BATCH_ID   = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc).isoformat()  # string from the start

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SAMPLE_MONTH : {SAMPLE_MONTH}")
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"BATCH_ID     : {BATCH_ID}")
print(f"STARTED_AT   : {STARTED_AT}")


## Step 2 — Schema definition

In [ ]:
# ---------------------------------------------------------------------------
# Empresas layout — 7 columns per official Receita Federal documentation
# Reference: https://arquivos.receitafederal.gov.br (layout PDF)
# Columns identified by position — CSV has NO headers
# ---------------------------------------------------------------------------
EMPRESAS_COLS = [
    "cnpj_basico",               # 0 — 8-digit CNPJ root
    "razao_social",              # 1 — legal name
    "natureza_juridica",         # 2 — legal nature code (FK rf_naturezas)
    "qualificacao_responsavel",  # 3 — responsible party qualification (FK rf_qualificacoes)
    "capital_social",            # 4 — share capital (Brazilian decimal: comma separator)
    "porte_empresa",             # 5 — company size code (01=ME, 03=EPP, 05=Demais)
    "ente_federativo_responsavel", # 6 — ~100% null (public entities only)
]

EXPECTED_COL_COUNT = len(EMPRESAS_COLS)  # drift = any shard with != 7 columns

TECHNICAL_COLUMNS = [
    "_source_id",
    "_batch_id",
    "_ingested_at",
    "_source_file",
    "_year_month",
]

ALL_COLUMNS = EMPRESAS_COLS + TECHNICAL_COLUMNS

# Column alias SQL fragment — built once, reused in every shard write
# Maps positional index to named column: "0" AS cnpj_basico, "1" AS razao_social, ...
COL_ALIASES = ", ".join(f'"{i}" AS {c}' for i, c in enumerate(EMPRESAS_COLS))

print(f"Source columns    : {len(EMPRESAS_COLS)}")
print(f"Technical columns : {len(TECHNICAL_COLUMNS)}")
print(f"Total columns     : {len(ALL_COLUMNS)}")
print(f"Expected per shard: {EXPECTED_COL_COUNT} source columns")


## Step 3 — Process shards

In [ ]:
def read_zip_csv(zip_path: Path, encoding: str = "latin-1") -> pd.DataFrame:
    """
    Read a Receita Federal CSV shard from a ZIP file into a pandas DataFrame.

    Uses pandas for ZIP/CSV reading because DuckDB does not provide the same
    encoding fallback mechanism needed for RF files (latin-1 / utf-8 / cp1252).

    Parameters
    ----------
    zip_path : path to the .zip file containing a single CSV
    encoding : primary encoding to try (default: latin-1)

    Returns
    -------
    pd.DataFrame
        All columns as string (dtype=str), no headers.

    Raises
    ------
    ValueError
        If none of the encodings succeed.
    """
    with zipfile.ZipFile(zip_path) as zf:
        csv_bytes = zf.read(zf.namelist()[0])

    for enc in [encoding, "utf-8", "cp1252"]:
        try:
            return pd.read_csv(
                io.BytesIO(csv_bytes),
                sep=";", header=None, dtype=str,
                encoding=enc, on_bad_lines="skip",
            )
        except (UnicodeDecodeError, Exception):
            continue

    raise ValueError(f"Could not decode {zip_path.name} with any known encoding")


def _log_drift_event(shard_name: str, actual_cols: int, expected_cols: int) -> None:
    """
    Append a schema drift event to schema_drift_log.json.

    For positional CSV sources, drift = unexpected column count.
    Column names cannot drift because they are assigned by position.

    Parameters
    ----------
    shard_name    : filename of the affected shard (e.g. "Empresas3.zip")
    actual_cols   : number of columns found in the shard
    expected_cols : expected column count (EXPECTED_COL_COUNT)
    """
    event = {
        "batch_id"      : BATCH_ID,
        "source_id"     : SOURCE_ID,
        "object"        : "bronze_rf_empresas",
        "event_type"    : "SCHEMA_DRIFT",
        "severity"      : "WARN",
        "action"        : "CONTINUE",
        "shard"         : shard_name,
        "expected_cols" : expected_cols,
        "actual_cols"   : actual_cols,
        "logged_at"     : datetime.now(timezone.utc).isoformat(),
    }
    existing = []
    if DRIFT_LOG.exists():
        with open(DRIFT_LOG, "r", encoding="utf-8") as f:
            try:
                existing = json.load(f)
            except json.JSONDecodeError:
                existing = []
    existing.append(event)
    DRIFT_LOG.parent.mkdir(parents=True, exist_ok=True)
    with open(DRIFT_LOG, "w", encoding="utf-8") as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)


# ---------------------------------------------------------------------------
# Discover shards
# ---------------------------------------------------------------------------
shard_files = sorted(SAMPLE_DIR.glob("Empresas*.zip"))

if not shard_files:
    raise FileNotFoundError(
        f"No Empresas*.zip files found in {SAMPLE_DIR}\n"
        "Run 01_bootstrap_receita_federal.py first."
    )

print(f"Shards found : {len(shard_files)}")
for s in shard_files:
    print(f"  {s.name} — {s.stat().st_size / 1e6:.1f} MB")
print()

# ---------------------------------------------------------------------------
# Process shards — append to single Parquet via atomic os.replace()
# ---------------------------------------------------------------------------
BRONZE_ROOT.mkdir(parents=True, exist_ok=True)
partition_dir = BRONZE_ROOT / f"_year_month={SAMPLE_MONTH}"
partition_dir.mkdir(parents=True, exist_ok=True)
output_file  = partition_dir / "data.parquet"
output_path  = to_sql_path(output_file)

# Idempotency: remove existing output before re-processing all shards
if output_file.exists():
    output_file.unlink()
    print(f"Removed existing: {output_file.name}")

con = get_connection()  # in-memory — Parquet I/O does not require persistent DB

results       = []
total_records = 0
has_drift     = False

for i, shard_path in enumerate(shard_files, 1):
    try:
        df          = read_zip_csv(shard_path)
        actual_cols = len(df.columns)

        # Schema drift — unexpected column count
        drift_flag = actual_cols != EXPECTED_COL_COUNT
        if drift_flag:
            has_drift = True
            _log_drift_event(shard_path.name, actual_cols, EXPECTED_COL_COUNT)
            print(f"  [DRIFT] {shard_path.name}: expected {EXPECTED_COL_COUNT} cols, got {actual_cols}")
            df = df.iloc[:, :EXPECTED_COL_COUNT]  # truncate to expected — extras ignored

        source_sql = to_sql_path(shard_path)

        if i == 1:
            # First shard: CREATE the Parquet file
            con.execute(f"""
                COPY (
                    SELECT
                        {COL_ALIASES},
                        '{SOURCE_ID}'    AS _source_id,
                        '{BATCH_ID}'     AS _batch_id,
                        CURRENT_TIMESTAMP AS _ingested_at,
                        '{source_sql}'   AS _source_file,
                        '{SAMPLE_MONTH}' AS _year_month
                    FROM df
                ) TO '{output_path}'
                (FORMAT PARQUET, COMPRESSION SNAPPY)
            """)
        else:
            # Subsequent shards: read existing + new shard, write combined atomically
            tmp_path = output_path + ".tmp"
            con.execute(f"""
                COPY (
                    SELECT * FROM read_parquet('{output_path}')
                    UNION ALL
                    SELECT
                        {COL_ALIASES},
                        '{SOURCE_ID}'    AS _source_id,
                        '{BATCH_ID}'     AS _batch_id,
                        CURRENT_TIMESTAMP AS _ingested_at,
                        '{source_sql}'   AS _source_file,
                        '{SAMPLE_MONTH}' AS _year_month
                    FROM df
                ) TO '{tmp_path}'
                (FORMAT PARQUET, COMPRESSION SNAPPY)
            """)
            os.replace(tmp_path, output_path)

        shard_records = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
        ).fetchone()[0]
        shard_count   = shard_records - total_records
        total_records = shard_records

        results.append({"shard": shard_path.name, "status": "SUCCESS",
                         "records": shard_count, "error": None})
        drift_flag_str = " [DRIFT]" if drift_flag else ""
        print(f"  [{i:02d}/{len(shard_files)}] OK {shard_path.name}"
              f" — {shard_count:>10,} rows — total: {total_records:>12,}{drift_flag_str}")

    except Exception as e:
        results.append({"shard": shard_path.name, "status": "ERROR",
                         "records": 0, "error": str(e)})
        print(f"  [{i:02d}/{len(shard_files)}] ERR {shard_path.name} — {str(e)[:80]}")

con.close()

error_count = sum(1 for r in results if r["status"] == "ERROR")
print()
print(f"Total records : {total_records:,}")
print(f"Shards OK     : {len(shard_files) - error_count}/{len(shard_files)}")
print(f"Errors        : {error_count}")
print(f"Schema drift  : {has_drift}")


## Step 4 — Validation

In [ ]:
suite   = CheckSuite("bronze_rf_empresas")
con_val = get_connection()

# Check 1 — no shard errors
suite.add(
    "No shard processing errors",
    error_count == 0,
    f"{error_count} error(s)",
)

# Check 2 — output Parquet exists
suite.add(
    "Output Parquet file exists",
    output_file.exists(),
    str(output_file),
)

# Check 3 — total records > 0
suite.add(
    "Total records written > 0",
    total_records > 0,
    f"{total_records:,}",
)

# Check 4 — record count matches sum of shard counts
bronze_total = con_val.execute(
    f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
).fetchone()[0]
suite.add(
    "Record count matches shard sum",
    bronze_total == total_records,
    f"parquet={bronze_total:,}  sum={total_records:,}",
)

# Check 5 — column count correct
actual_cols = con_val.execute(
    f"SELECT COUNT(*) FROM (DESCRIBE SELECT * FROM read_parquet('{output_path}') LIMIT 0)"
).fetchone()[0]
suite.add(
    "Column count correct",
    actual_cols == len(ALL_COLUMNS),
    f"found={actual_cols}  expected={len(ALL_COLUMNS)}",
)

# Check 6 — all technical columns present
existing_cols = {
    row[0] for row in con_val.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{output_path}') LIMIT 0"
    ).fetchall()
}
missing_tech = [c for c in TECHNICAL_COLUMNS if c not in existing_cols]
suite.add(
    "All technical columns present",
    not missing_tech,
    f"missing: {missing_tech}" if missing_tech else "all present",
)

# Check 7 — no zero-record shards
zero_shards = [r["shard"] for r in results if r["status"] == "SUCCESS" and r["records"] == 0]
suite.add(
    "No zero-record shards",
    not zero_shards,
    f"empty: {zero_shards}" if zero_shards else "all shards contributed records",
)

# Check 8 — cnpj_basico always 8 digits
non_8digit = con_val.execute(
    f"SELECT COUNT(*) FROM read_parquet('{output_path}') "
    "WHERE cnpj_basico IS NOT NULL AND LENGTH(cnpj_basico) != 8"
).fetchone()[0]
suite.add(
    "cnpj_basico always 8 digits",
    non_8digit == 0,
    f"{non_8digit:,} rows with wrong length",
)

# Check 9 — schema drift monitored (always PASS)
suite.add(
    "Schema drift monitored",
    True,
    "drift detected — check schema_drift_log.json" if has_drift else "none detected",
)

con_val.close()  # close before potential raise
suite.report()
suite.assert_all_pass()


## Step 5 — Ops registration

In [ ]:
bytes_written = output_file.stat().st_size if output_file.exists() else 0
errors_detail = (
    "; ".join(f"{r['shard']}: {r['error'][:60]}" for r in results if r["status"] == "ERROR")
    if error_count > 0 else None
)

entry = make_entry(
    source_id     = SOURCE_ID,
    period        = SAMPLE_MONTH,
    status        = "SUCCESS" if error_count == 0 else "PARTIAL",
    records       = total_records,
    bytes_written = bytes_written,
    started_at    = STARTED_AT,
    finished_at   = datetime.now(timezone.utc).isoformat(),
    error_message = errors_detail,
    # Bronze-specific extra fields
    batch_id         = BATCH_ID,
    layer            = "bronze",
    object           = "rf_empresas",
    files            = len(shard_files),
    has_rescued_data = has_drift,
    drift_months     = 1 if has_drift else 0,
)

append_log(LOG_PATH, entry)

print(f"Batch registered   : {BATCH_ID}")
print(f"Status             : {entry['status']}")
print(f"Records            : {total_records:,}")
print(f"Bytes written      : {bytes_written / (1024*1024):.1f} MB")
print(f"Has rescued data   : {has_drift}")
print(f"Log                : {LOG_PATH}")
